# Fact Set Composition — Gold Layer

Pre-aggregated fact table at the grain of one row per set. Provides composition metrics like total parts, unique colors, dominant color, dominant part category, and minifig count.

## What this notebook does

1. **Read** — Loads `fct_set_inventory`, `fct_set_minifigs`, and `dim_set` from gold.
2. **Transform** — Aggregates per-set metrics: total/unique parts, unique colors, transparent %, spare %, dominant color, dominant part category, and minifig count.
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/fct_set_composition`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.fct_set_composition`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, sum as _sum, countDistinct, round as _round,
    when, row_number, first, coalesce, lit,
)

spark = SparkSession.builder.getOrCreate()

GOLD_FCT_SET_INVENTORY = "lego_brickbase.gold.fct_set_inventory"
GOLD_FCT_SET_MINIFIGS  = "lego_brickbase.gold.fct_set_minifigs"
DELTA_TABLE_PATH       = "/Volumes/lego_brickbase/gold/delta_volume/fct_set_composition"
CATALOG_TABLE          = "lego_brickbase.gold.fct_set_composition"

## Read and Transform

Aggregate inventory-level detail into per-set composition metrics.

In [ ]:
df_inv = spark.table(GOLD_FCT_SET_INVENTORY)
df_minifigs = spark.table(GOLD_FCT_SET_MINIFIGS)

# Core metrics per set
df_set_metrics = (
    df_inv
    .groupBy("set_key", "set_number", "set_name", "year", "theme_name", "root_theme_name")
    .agg(
        _sum("quantity").alias("total_parts"),
        countDistinct("part_key").alias("total_unique_parts"),
        countDistinct("color_key").alias("total_unique_colors"),
        _sum(when(col("is_transparent_color") == True, col("quantity")).otherwise(0)).alias("transparent_part_quantity"),
        _sum(when(col("is_spare") == True, col("quantity")).otherwise(0)).alias("spare_part_quantity"),
    )
)

df_set_metrics = df_set_metrics.withColumn(
    "pct_transparent_parts",
    _round(col("transparent_part_quantity") / col("total_parts") * 100, 2),
).withColumn(
    "pct_spare_parts",
    _round(col("spare_part_quantity") / col("total_parts") * 100, 2),
)

# Dominant color per set (color with highest total quantity)
w_color = Window.partitionBy("set_key").orderBy(col("color_qty").desc())
df_dominant_color = (
    df_inv
    .groupBy("set_key", "color_name")
    .agg(_sum("quantity").alias("color_qty"))
    .withColumn("rn", row_number().over(w_color))
    .filter(col("rn") == 1)
    .select(
        col("set_key").alias("_dc_set_key"),
        col("color_name").alias("dominant_color_name"),
    )
)

# Dominant part category per set
w_cat = Window.partitionBy("set_key").orderBy(col("cat_qty").desc())
df_dominant_cat = (
    df_inv
    .groupBy("set_key", "part_category_name")
    .agg(_sum("quantity").alias("cat_qty"))
    .withColumn("rn", row_number().over(w_cat))
    .filter(col("rn") == 1)
    .select(
        col("set_key").alias("_dcat_set_key"),
        col("part_category_name").alias("dominant_part_category"),
    )
)

# Minifig count per set
df_minifig_agg = (
    df_minifigs
    .groupBy("set_key")
    .agg(_sum("quantity").alias("number_of_minifigs"))
    .select(
        col("set_key").alias("_mf_set_key"),
        col("number_of_minifigs"),
    )
)

# Join all together
df_fct = (
    df_set_metrics
    .join(df_dominant_color, df_set_metrics["set_key"] == df_dominant_color["_dc_set_key"], "left")
    .join(df_dominant_cat, df_set_metrics["set_key"] == df_dominant_cat["_dcat_set_key"], "left")
    .join(df_minifig_agg, df_set_metrics["set_key"] == df_minifig_agg["_mf_set_key"], "left")
    .select(
        df_set_metrics["set_key"],
        col("set_number"),
        col("set_name"),
        col("year"),
        col("theme_name"),
        col("root_theme_name"),
        col("total_parts"),
        col("total_unique_parts"),
        col("total_unique_colors"),
        col("pct_transparent_parts"),
        col("pct_spare_parts"),
        col("dominant_color_name"),
        col("dominant_part_category"),
        coalesce(col("number_of_minifigs"), lit(0)).alias("number_of_minifigs"),
    )
)

df_fct.printSchema()
df_fct.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_fct
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        set_key                STRING  NOT NULL COMMENT 'Foreign key to dim_set; surrogate key identifying the set',
        set_number             STRING           COMMENT 'Official LEGO set number (e.g. 75192-1)',
        set_name               STRING           COMMENT 'Official name of the LEGO set',
        year                   INTEGER          COMMENT 'Year the set was released',
        theme_name             STRING           COMMENT 'Name of the direct theme the set belongs to',
        root_theme_name        STRING           COMMENT 'Name of the top-level ancestor theme',
        total_parts            BIGINT           COMMENT 'Total sum of all individual part quantities in the set',
        total_unique_parts     BIGINT           COMMENT 'Number of distinct part designs in the set',
        total_unique_colors    BIGINT           COMMENT 'Number of distinct colors used across all parts in the set',
        pct_transparent_parts  DOUBLE           COMMENT 'Percentage of total part quantity that consists of transparent parts (0–100)',
        pct_spare_parts        DOUBLE           COMMENT 'Percentage of total part quantity that are spare parts (0–100)',
        dominant_color_name    STRING           COMMENT 'Name of the color that has the highest total part quantity in this set',
        dominant_part_category STRING           COMMENT 'Part category with the highest total part quantity in this set',
        number_of_minifigs     BIGINT           COMMENT 'Total number of minifigures included in this set',
        CONSTRAINT fct_set_composition_pk
            PRIMARY KEY (set_key),
        CONSTRAINT fct_set_composition_fk_set
            FOREIGN KEY (set_key) REFERENCES lego_brickbase.gold.dim_set (set_key)
    )
    COMMENT 'Pre-aggregated fact table at the set grain. Provides composition metrics including part counts, color diversity, transparency and spare percentages, dominant color/category, and minifig count.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")